# 12.6 - LangChain Tools
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
LLMs cannot compute or look things up reliably by themselves. Tools are Python functions decorated
with `@tool` that the model can call. This is the bridge from "chat" to "agent".
## 2. Why Does This Matter?
Tools give the model real actions (calculations, lookups, queries). This unit is the foundation for
Phase 13/14 agent loops.
## 3. Prerequisites
- Unit 12.4 (chains)
- Python functions, type hints, docstrings
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Define tools with `@tool` + helpful docstrings
- Bind tools to a model with `model.bind_tools(tools)`
- Run a manual tool-calling loop (max 3 iterations)
- Handle a missing/incorrect tool call with a fallback message
## 5. Mental Model
A tool is a documented function the model *chooses* to call. The model returns a `tool_calls`
request; you execute it and feed the result back as a `ToolMessage`; the model then gives a final
answer.

```text
user question -> model(bound tools) -> tool_calls? -> execute tool -> ToolMessage -> model -> final answer
                                     \-> no tool_calls -> answer directly


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq


## 7. Define Tools With @tool
The docstring becomes the tool's instruction manual for the model. Keep each tool to one job and
make the return JSON-serializable (a number or string).

In [3]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together. Use for any arithmetic product."""
    return round(a * b, 4)


@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between km, miles, kg, lbs. Returns a human string."""
    table = {"km": 1.0, "miles": 0.621371, "kg": 1.0, "lbs": 2.20462}
    if from_unit not in table or to_unit not in table:
        return f"cannot convert {from_unit} -> {to_unit}"
    return f"{value} {from_unit} = {round(value * table[to_unit] / table[from_unit], 4)} {to_unit}"


@tool
def greeting(name: str) -> str:
    """Return a friendly greeting for a person's name (no computation)."""
    return f"Hello, {name}!"

for t in (multiply, convert_units, greeting):
    print(t.name, "|", t.description[:50])


multiply | Multiply two numbers together. Use for any arithme
convert_units | Convert between km, miles, kg, lbs. Returns a huma
greeting | Return a friendly greeting for a person's name (no


## 8. Bind Tools to the Model
`bind_tools` tells the model what is available without calling anything yet.

In [4]:
TOOLS = [multiply, convert_units, greeting]
print("bound tool names:", [t.name for t in TOOLS])


bound tool names: ['multiply', 'convert_units', 'greeting']


## 9. Manual Tool-Calling Loop (max 3 iterations)
The core agent pattern: call the model -> read `tool_calls` -> execute -> append `ToolMessage` ->
call again -> stop when the model answers without a tool call. Offline we substitute a deterministic
"fake model" that picks a tool by keyword so the whole loop still runs.

In [5]:
import re


def mock_model(messages, tools):
    """Deterministic stand-in for the LLM when offline / on error."""
    has_tool_result = any(isinstance(m, ToolMessage) for m in messages)
    if has_tool_result:
        return AIMessage(content="mock final answer: done, result was " + messages[-1].content)
    text = " ".join(str(getattr(m, "content", "")) for m in messages).lower()
    if "multiply" in text or "times" in text or "*" in text:
        m2 = re.search(r"(\d+)\s*(?:and|x|\*)\s*(\d+)", text)
        a, b = (float(m2.group(1)), float(m2.group(2))) if m2 else (6.0, 7.0)
        return AIMessage(content="call multiply", tool_calls=[{"name": "multiply",
                          "args": {"a": a, "b": b}, "id": "mock-1"}])
    if "convert" in text or "km" in text or "miles" in text:
        m2 = re.search(r"(\d+(?:\.\d+)?)\s*(km|miles|kg|lbs)", text)
        v, u = (float(m2.group(1)), m2.group(2)) if m2 else (10.0, "km")
        return AIMessage(content="call convert_units", tool_calls=[{"name": "convert_units",
                          "args": {"value": v, "from_unit": u, "to_unit": "miles"}, "id": "mock-2"}])
    return AIMessage(content="mock: no tool needed")


def call_model(messages, tools, max_iter=3):
    if not os.environ.get("GROQ_API_KEY"):
        return mock_model(messages, tools)
    try:
        chat = ChatGroq(model=GROQ_MODEL, temperature=0.0).bind_tools(tools)
        return chat.invoke(messages)
    except Exception as e:
        return mock_model(messages, tools)


tool_map = {t.name: t for t in TOOLS}


def run_agent(question: str, max_iter=3):
    messages = [HumanMessage(content=question)]
    history = list(messages)
    for step in range(1, max_iter + 1):
        ai = call_model(history, TOOLS)
        history.append(ai)
        calls = getattr(ai, "tool_calls", None) or []
        if not calls:
            print(f"iter {step}: final answer ->", ai.content)
            return ai.content
        for call in calls:
            name, args = call["name"], call.get("args") or {}
            print(f"iter {step}: model wants tool '{name}' with args {args}")
            if name not in tool_map:
                msg = f"tool {name} is not available"
                fallback = ToolMessage(content=msg, tool_call_id=call["id"])
            else:
                try:
                    result = tool_map[name].invoke(args)
                    fallback = ToolMessage(content=str(result), tool_call_id=call["id"])
                except Exception as e:
                    fallback = ToolMessage(content=f"tool error: {type(e).__name__}",
                                           tool_call_id=call["id"])
            history.append(fallback)
            print(f"       tool returned -> {fallback.content}")
    print("max iterations reached without a final answer; using last assistant text.")
    return history[-1].content


run_agent("What is 7 times 6?")
print("---")
run_agent("Convert 10 km to miles.")
print("---")
run_agent("Hello there!")


iter 1: model wants tool 'multiply' with args {'a': 7, 'b': 6}
       tool returned -> 42.0


iter 2: final answer -> 42.0
---


iter 1: model wants tool 'convert_units' with args {'from_unit': 'km', 'to_unit': 'miles', 'value': 10}
       tool returned -> 10.0 km = 6.2137 miles


iter 2: final answer -> 10 km is approximately **6.2137 miles**.
---


iter 1: model wants tool 'greeting' with args {'name': 'there'}
       tool returned -> Hello, there!


iter 2: final answer -> Hello! How can I help you today?


'Hello! How can I help you today?'

## 10. Missing Tool Args → Fallback Message
The model may forget an argument or request a tool that does not exist. We catch both and return a
fallback `ToolMessage` instead of crashing.

In [6]:
@tool
def divide(a: float, b: float) -> str:
    """Divide a by b. Returns a string so division-by-zero stays graceful."""
    if b == 0:
        return "error: division by zero"
    return str(a / b)


DIV_TOOLS = [divide]
placeholder_msg = HumanMessage(content="Divide 10 by 2")
ai_bad = AIMessage(content="call divide", tool_calls=[{"name": "divide", "args": {"a": 10}, "id": "x1"}])
# missing 'b' arg -> tool_map invoke errors -> fallback message
try:
    result = divide.invoke(ai_bad.tool_calls[0]["args"])
    print("invoked with partial args:", result)
except Exception as e:
    fb = ToolMessage(content=f"tool error: {type(e).__name__}", tool_call_id="x1")
    print("missing-arg fallback ->", fb.content)

# unknown tool name -> graceful fallback
fb2 = ToolMessage(content="tool 'hack_db' is not available", tool_call_id="x2")
print("unknown-tool fallback ->", fb2.content)


missing-arg fallback -> tool error: ValidationError
unknown-tool fallback -> tool 'hack_db' is not available


## 11. Why Tools Are Needed (concept recap)
A pure model has no arithmetic consistency. Compare a direct string guess vs the tool result.

In [7]:
print('model-style guess : "42" (unreliable)')
print("tool result       :", multiply.invoke({"a": 7, "b": 6}), "(exact, auditable)")


model-style guess : "42" (unreliable)
tool result       : 42.0 (exact, auditable)




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Vague tool descriptions — the model literally reads them to decide when to call.
- No error handling inside tools (a failing tool should return a message, not raise).
- Forgetting to feed the `ToolMessage` back to the model before asking for the final answer.
- Tools that do too many things (keep one job each).

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Model never calls a tool | Description unclear | Add examples to the docstring |
| Tool call fails | Missing required arg | Validate/fallback in the loop |
| Model ignores tool result | ToolMessage not fed back / wrong id | Append result with correct `tool_call_id` |
| Wrong return type | Signature mismatch | Return JSON-serializable str/number |

### Best Practices (applied)

- Write specific docstrings — they are read by the model.
- Validate inputs inside tools.
- Return error strings, never raise, from tools.
- Keep tools single-purpose.

### Hands-On Practice

1. **Basic:** Call `multiply.invoke` with two numbers directly.
2. **Guided:** Bind `divide` and verify the fallback on a zero divisor.
3. **Independent:** Add a `string_reverse` tool and run it through the loop.
4. **Realistic:** Ask ambiguous questions and verify the model picks a sensible tool.
5. **Challenge:** Add a 4th tool and confirm the loop routes to it.

### Exit Criteria

- You can define and use LangChain tools.
- You can bind tools to an LLM and verify tool calls.
- You can handle tool execution errors.
